# OOP Pillars & Property Accessors

The four pillars of OOP are encapsulation, abstraction, inheritance, and polymorphism. Inheritance is covered separately in [[JS - Classes, Objects and Inheritance]]; this note handles the other three plus getters and setters.

Related: [[JS - The new Operator]] · [[JS - Functional Constructors and Errors]]

---

## 1. Encapsulation

Bundling data and the code that operates on it into one unit, while restricting direct access to internal state. Since ES2022, JavaScript enforces this with `#` private fields.

```js
class BankAccount {
  #balance = 0;  // private — inaccessible outside the class body

  deposit(amount) {
    if (amount > 0) this.#balance += amount;
  }

  getBalance() {
    return this.#balance;  // controlled public access
  }
}

const account = new BankAccount();
account.deposit(100);
account.getBalance();   // 100
// account.#balance;    // SyntaxError, not a runtime error
```

Note that this is a **syntax** error — the code won't even parse. Contrast with `_balance`, which is pure convention and does nothing.

### The full private surface

Fields aren't the only thing that can be private:

```js
class Machine {
  #state = 'idle';           // private field
  static #count = 0;         // private static field
  #reset() { ... }           // private method
  get #status() { ... }      // private getter
  static #make() { ... }     // private static method
}
```

### Three things people trip on

**Private fields are not inherited.** A subclass cannot see the parent's `#balance`. If a child needs access, expose a `protected`-style accessor — JavaScript has no `protected`, so that's just a normal (or conventionally-underscored) method.

**They're invisible to almost everything.** `JSON.stringify`, `Object.keys`, spread, and `for...in` all skip them. That's usually what you want, but it means serialisation needs an explicit `toJSON()`:

```js
class BankAccount {
  #balance = 0;
  toJSON() { return { balance: this.#balance }; }
}
```

**They double as a brand check.** `in` works with private names and never throws:

```js
static isAccount(obj) { return #balance in obj; }
```

This is more reliable than `instanceof`, which breaks across realms (iframes, `vm` contexts).

### The pre-ES2022 patterns

Worth recognising in older code:

```js
// Closure-based — genuinely private, but methods are per-instance
function createAccount() {
  let balance = 0;
  return {
    deposit(amt) { if (amt > 0) balance += amt; },
    getBalance() { return balance; }
  };
}

// WeakMap-based — keeps methods on the prototype
const _balance = new WeakMap();
class BankAccount {
  constructor() { _balance.set(this, 0); }
  getBalance() { return _balance.get(this); }
}
```

---

## 2. Getters and Setters

Accessors intercept property reads and writes, so you can run validation or computation while the call site still looks like plain property access.

- **`get`** — a property whose value is computed or safely retrieved.
- **`set`** — validates or transforms data before storing it.

```js
class User {
  #name;

  constructor(name) {
    this.name = name;   // goes through the setter — validation on construction too
  }

  get name() {
    return this.#name.toUpperCase();
  }

  set name(newName) {
    if (newName.length < 3) throw new RangeError('Name is too short');
    this.#name = newName;
  }
}

const user = new User("Alex");
user.name;          // "ALEX" — read like a property, no parentheses
user.name = "Bo";   // RangeError
user.name = "John";
user.name;          // "JOHN"
```

I've changed two things from the common textbook version. The backing field is `#name` rather than `_name`, because the underscore is decorative — anyone can still write `user._name = "Bo"` and skip the setter entirely. And the setter **throws** instead of logging and returning; a silent no-op assignment is a debugging nightmare, since `user.name = "Bo"` looks like it worked.

### The infinite recursion trap

The single most common accessor bug:

```js
class User {
  get name() { return this.name; }   // ⚠️ calls itself forever
  set name(v) { this.name = v; }     // ⚠️ same
}
// RangeError: Maximum call stack size exceeded
```

An accessor must never read or write the property it is named after. Always use a differently-named backing field: `#name`, `_name`, or a WeakMap.

### Where accessors live

Class accessors go on the **prototype** and are non-enumerable. That has a visible consequence:

```js
JSON.stringify(user);   // "{}"  — the getter isn't an own enumerable property
```

The private field isn't serialised and neither is the prototype getter, so you get an empty object. Add `toJSON()` if you need output.

Outside classes, the same feature exists in object literals and via `Object.defineProperty`:

```js
const obj = {
  first: 'Ada', last: 'Lovelace',
  get full() { return `${this.first} ${this.last}`; }
};

Object.defineProperty(obj, 'id', {
  get() { return this._id; },
  enumerable: false,
  configurable: false
});
```

### When to use them

Good: validation on write, computed/derived values, lazy initialisation, keeping a public API stable while the internals change, read-only properties (getter with no setter).

Bad: anything expensive or side-effecting. A getter looks like a field at the call site, so a reader assumes it's cheap. If it hits the network or does heavy work, make it a method with parentheses. A setter with no getter is also a trap — reading the property returns `undefined`.

---

## 3. Polymorphism

"Many forms" — different classes responding to the same method name with their own behaviour. In class-based JS this is method overriding:

```js
class Animal {
  makeSound() { console.log("Some generic animal sound"); }
}
class Dog extends Animal {
  makeSound() { console.log("Woof! Woof!"); }
}
class Cat extends Animal {
  makeSound() { console.log("Meow!"); }
}

const animals = [new Dog(), new Cat(), new Animal()];
animals.forEach(a => a.makeSound());
// Woof! Woof!
// Meow!
// Some generic animal sound
```

The caller doesn't branch on type. Adding a `Cow` class requires no change to the loop — that's the actual payoff, not the polymorphism itself.

### Duck typing — polymorphism without inheritance

JavaScript is dynamically typed, so shared behaviour needs no shared ancestor. If it responds to `makeSound()`, it works:

```js
const robot = { makeSound: () => console.log("Beep boop") };
[new Dog(), robot].forEach(a => a.makeSound());  // both fine
```

This is the more idiomatic JS approach. The `extends` version is only necessary when there's genuinely shared implementation to reuse.

### There is no method overloading

Unlike Java or C#, you cannot define two methods with the same name and different signatures — the second declaration simply replaces the first. Simulate it with default parameters, rest args, or an options object:

```js
class Rect {
  constructor(width, height = width) {   // one arg → square
    this.width = width;
    this.height = height;
  }
}
```

### Polymorphism with built-in protocols

Overriding well-known methods makes your objects behave polymorphically with the *language*, not just your own code:

```js
class Money {
  constructor(amount) { this.amount = amount; }
  toString() { return `₹${this.amount}`; }
  valueOf() { return this.amount; }
  toJSON() { return { amount: this.amount }; }
  [Symbol.iterator]() { ... }
}

`${new Money(500)}`;        // "₹500"   — uses toString
new Money(500) + 100;       // 600      — uses valueOf
JSON.stringify(new Money(500));  // {"amount":500}
```

### Liskov substitution, briefly

A subclass should be usable anywhere the parent is, without surprising the caller. If `Dog.makeSound()` throws when `Animal.makeSound()` doesn't, or a `Square extends Rectangle` breaks code that sets width and height independently, the hierarchy is wrong. When overriding, keep the contract: accept at least what the parent accepts, return no less than it returns.

---

## 4. Abstraction

Hiding implementation detail and exposing only what the caller needs. JavaScript has no `abstract` keyword, so it's achieved with private methods and runtime guards.

```js
class CoffeeMachine {
  // The entire public interface
  brewCoffee() {
    this.#heatWater();
    this.#grindBeans();
    console.log("Your coffee is ready! ☕");
  }

  // Hidden internals
  #heatWater()  { console.log("Heating water to 95°C..."); }
  #grindBeans() { console.log("Grinding coffee beans..."); }
}

new CoffeeMachine().brewCoffee();
```

The user calls one method. The steps can be reordered, renamed, or rewritten without breaking anyone.

### Simulating abstract classes

Two guards, both worth having:

```js
class Shape {
  constructor() {
    if (new.target === Shape) {
      throw new TypeError('Shape is abstract and cannot be instantiated');
    }
  }

  area() {
    throw new Error(`${this.constructor.name} must implement area()`);
  }
}

class Circle extends Shape {
  constructor(r) { super(); this.r = r; }
  area() { return Math.PI * this.r ** 2; }
}

new Shape();            // TypeError
new Circle(2).area();   // 12.566...
```

`new.target === Shape` blocks direct instantiation while allowing subclasses. The throwing `area()` documents the required method and fails loudly if a subclass forgets it.

For fail-fast at construction rather than at call time, verify the shape of the subclass:

```js
class Shape {
  constructor() {
    if (new.target === Shape) throw new TypeError('Shape is abstract');
    for (const m of ['area', 'perimeter']) {
      if (typeof this[m] !== 'function') {
        throw new TypeError(`${new.target.name} must implement ${m}()`);
      }
    }
  }
}
```

TypeScript's `abstract class` and `interface` give you all of this at compile time with no runtime cost — the usual reason people reach for it once a codebase grows.

---

## Quick Reference

| Pillar | Mechanism in JS |
|---|---|
| Encapsulation | `#private` fields and methods, closures, WeakMaps |
| Abstraction | Private methods, `new.target` guard, throwing base methods |
| Inheritance | `extends` + `super`, prototype chain, mixins |
| Polymorphism | Method overriding, duck typing, `toString`/`valueOf`/`Symbol.*` |

---

## References

- [MDN — Using classes](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Using_classes)
- [MDN — Private properties](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes/Private_properties)
- [MDN — get](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Functions/get) · [set](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Functions/set)
- [MDN — Object.defineProperty](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Object/defineProperty)
- [javascript.info — Property getters and setters](https://javascript.info/property-accessors)